# 04 · Advanced Customization — the Stochastic Background (SGWB)

The headline of this workshop. gwmock has a **first-class SGWB signal**
backend (`source-type: sgwb`) that draws a correlated, coloured background
across a detector network from a power-law energy-density spectrum.

We'll generate it, **check it against theory**, see the cross-detector
correlation, and sketch the two extension paths: a **custom backend** and
the **`sgwbjax`** inference companion.

<!-- Replace USER/REPO/BRANCH after pushing; Isaac will wire the real Colab links. -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USER/REPO/blob/main/materials/notebooks/04-advanced-sgwb.ipynb)

In [ ]:
# === Run this ONCE ===
# Installs gwmock on Google Colab / fresh kernels (needs Python 3.12-3.14).
# If gwmock is already available (e.g. you pre-installed it into a local
# venv), this skips the install and just reports the version.
# The [sgwb] extra pulls in correlated-noise support used in notebook 04.
try:
    import gwmock
    print("gwmock already available:", gwmock.__version__)
except ModuleNotFoundError:
    %pip install -q "gwmock[sgwb]"

## 1. The model

gwmock parametrises the background as a power law in the GW energy density:

$$\Omega_{\rm GW}(f) = \Omega_{\rm ref}\left(\frac{f}{f_{\rm ref}}\right)^{\alpha}$$

which maps to a one-sided **strain** PSD

$$S_h(f) = \frac{3 H_0^2}{10\pi^2}\,\frac{\Omega_{\rm GW}(f)}{f^3}.$$

Config keys (under `orchestration.signal.parameters`):
`omega_ref` (required), `spectral_index` (= $\alpha$, default 0),
`reference_frequency` (= $f_{\rm ref}$ in Hz, default 25).

## 2. Generate a small SGWB dataset

512 Hz, 16 s — runs in about a second. `source-type: sgwb` needs **no**
population block (it's not a catalogue of events, it's a field).

In [ ]:
%%writefile sgwb.yaml
globals:
    simulator-arguments:
        sampling-frequency: 512
        duration: 16
        total-duration: 16
        start-time: 1577491218
        seed: 42
    working-directory: .
    output-directory: output
    metadata-directory: metadata
orchestration:
    signal:
        source-type: sgwb
        detectors: [ET-2L-Aligned]
        minimum-frequency: 5
        parameters:
            omega_ref: 1.0e-9
            spectral_index: 0.0
            reference_frequency: 25.0
        output:
            output_directory: signal
            file_name: sgwb-{{ counter }}.hdf5
            arguments: {channel: '{{ detectors }}:SGWB'}

In [ ]:
!gwmock simulate sgwb.yaml 2>&1 | tail -2

## 3. Does the data match the model? (anchor to theory)

Always check a generator against an independent calculation. We estimate
the PSD of the generated strain (Welch) and overlay $S_h(f)$ computed from
the **same** $\Omega_{\rm ref}, \alpha, f_{\rm ref}$.

In [ ]:
import h5py, numpy as np, matplotlib.pyplot as plt
from scipy.signal import welch

with h5py.File('output/signal/sgwb-0.hdf5') as f:
    ds = list(f.keys()); x = f[ds[0]][:]
fs = 512.0
omega_ref, alpha, fref = 1.0e-9, 0.0, 25.0
H0 = 67.66 * 1000 / 3.0856775814913673e22   # km/s/Mpc -> 1/s (Planck-18)

fr, pxx = welch(x, fs=fs, nperseg=2048, detrend=False)
def Sh(fq):
    out = np.zeros_like(fq); m = fq > 0
    out[m] = 3*H0**2/(10*np.pi**2) * omega_ref*(fq[m]/fref)**alpha / fq[m]**3
    return out
band = (fr >= 8) & (fr <= 200)
print('median measured/theory PSD ratio (8-200 Hz): %.2f' % np.nanmedian(pxx[band]/Sh(fr)[band]))

plt.loglog(fr[fr>0], pxx[fr>0], lw=0.6, label='measured (Welch)')
plt.loglog(fr[fr>0], Sh(fr)[fr>0], 'k--', label=r'theory $S_h(f)$')
plt.xlim(5, 256); plt.xlabel('frequency [Hz]'); plt.ylabel(r'$S_h(f)$ [1/Hz]')
plt.legend(); plt.title('SGWB strain PSD vs theory'); plt.grid(True, which='both', alpha=0.3); plt.show()

The median ratio sits near **1** (a single 16 s realisation scatters bin
to bin — that's cosmic/sample variance, not a bug; average more or use
longer data to tighten it). The generator faithfully reproduces the model.

### Try it
- Set `spectral_index: 0.667` (the astrophysical CBC background slope) and
  re-run — the PSD tilts and the overlay should still track.
- Raise `omega_ref` to `1.0e-8` — the whole curve scales up 10×.

## 4. Cross-detector correlation — the point of an SGWB

A stochastic background is **correlated** between detectors (set by the
overlap reduction function), unlike instrument noise. gwmock builds the
cross-spectral density across the network, so the two ET-2L channels are
genuinely correlated:

In [ ]:
with h5py.File('output/signal/sgwb-0.hdf5') as f:
    ks = list(f.keys()); a = f[ks[0]][:]; b = f[ks[1]][:]
print('channels:', ks)
print('Pearson correlation between the two ET-2L channels: %.3f' % np.corrcoef(a, b)[0,1])
print('(instrument noise between independent detectors would be ~0)')

## 5. Extension path A — a custom signal backend

Need a spectrum gwmock doesn't ship (broken power law, peaked, a specific
cosmological model)? Implement the `GWSimulator` protocol and register it.
Then any config can select it by name. Sketch:

```python
from gwmock_signal import GWSimulator, register_simulator_backend

class MySgwbSimulator(GWSimulator):
    @property
    def required_params(self):
        return frozenset({'omega_ref', 'f_break'})

    def simulate(self, params, detector_names, background=None, *,
                 sampling_frequency, minimum_frequency,
                 earth_rotation=True, interpolate_if_offset=True):
        # build your S_h(f), cross-spectral matrix, draw correlated strain,
        # return a DetectorStrainStack
        ...

register_simulator_backend('my_sgwb', MySgwbSimulator)
# config:  orchestration.signal.source-type: my_sgwb
```

Third-party packages can also register via a `gwmock.signal` entry point,
so installing your package makes the backend available with no code edits.
See `gwmock-signal`'s *custom backends* guide for the full contract.

## 6. Extension path B — inference with `sgwbjax`

gwmock **generates**; [`sgwbjax`](https://github.com/Leuven-Gravity-Institute)
**infers**. It reads a gwmock dataset straight into Whittle-likelihood
Fourier segments for SGWB parameter estimation (JAX + BlackJAX):

```python
from sgwbjax.datasets import import_gwmock_dataset
ds = import_gwmock_dataset('output/signal/sgwb-0.hdf5', segment_duration_s=4.0)
# ds.strain_segments : complex RFFT coeffs (n_seg, n_freq, n_det)
# ds.frequencies_hz, ds.detectors, ds.truth  -> feed the likelihood
```

`sgwbjax` is a separate install (it pulls in JAX) and out of scope to run
here, but this is the natural next step from generated data to a posterior
on $\Omega_{\rm ref}$ and $\alpha$.

---
## You've completed the workshop 🎉

You can now: install gwmock · write & run configs · read the provenance
metadata and reproduce a run · merge to analysis strain · generate and
**validate** an SGWB background · and extend it with a custom backend.

Docs: <https://leuven-gravity-institute.github.io/gwmock>